# E9.5 cell-state ATAC tracks

Builds misha ATAC tracks of the E9.5 head parts per annotated cell state, separately for host and DKO.

Inputs:
- single-cell ATAC counts, `cell_to_metacell.csv`, and the MCProj projection `dnmt_dko_brain_metacells_gotg95_projection.h5ad`, all under `base_dir` (`e95_dir` in `paths.R`)
- reannotation `revision_e95/data/annot_2909.csv`

Outputs:
- `out_path("./data/md_metacells.csv")`: metacell metadata (read by `prep/E9.5_atac_peaks.ipynb`, `Fig6_func.r`)
- shared misha database (not `git_test/`): `embdnmt.brain_e95_{host,dko}.reannot.<cell_type>`, host `Sox2_pool` and `Twist1_pool`, `embdnmt.brain_e95_dko.marginal_plus_gastr`

Caveat:
- The track-writing cells were not run in the stored notebook; the tracks exist and are not overwritten, so re-running fails unless they are removed first.

## Setup

Loads mcATAC and misha (mm10), the development mcATAC from `dev_src_dir`, and sets `base_dir`.

In [ ]:
source("../paths.R")
setwd(paste0(embdnmt_dir, "/revision_e95"))

In [1]:
library(tidyverse)
library(mcATAC)
library(tgutil)
library(here)
library(misha.ext)
gset_genome("mm10")

In [3]:
library(mcATAC)

In [11]:
base_dir = paste0(e95_dir, "/")

In [35]:
devtools::load_all(paste0(dev_src_dir, "/mcATAC"))

## Metacell ATAC counts

`scc_to_mcc_multi_batch` sums single-cell ATAC counts per metacell (joint host + DKO assignment; metacells `-1`, `-2` dropped), for host (`mcc_host`) and DKO (`mcc_dko`).

In [12]:
cell_to_metacell <- read_csv(paste0(base_dir, "data/cell_to_metacell.csv"))

In [13]:
mcc_both <- scc_to_mcc_multi_batch(scc_dirs = c(paste0(base_dir,"raw/mm_embryo/dnmt_dko_brain/scc_counts"), paste0(base_dir,"raw/mm_embryo/dnmt_dko_brain_host/scc_counts")), cell_to_metacell, ignore_metacells = c(-1, -2))
mcc_host <- scc_to_mcc_multi_batch(scc_dirs = c(paste0(base_dir,"raw/mm_embryo/dnmt_dko_brain_host/scc_counts")), cell_to_metacell %>% filter(source == "host"), ignore_metacells = c(-1, -2))
mcc_dko <- scc_to_mcc_multi_batch(scc_dirs = c(paste0(base_dir,"raw/mm_embryo/dnmt_dko_brain/scc_counts")), cell_to_metacell %>% filter(source == "dko"), ignore_metacells = c(-1, -2))

## Metacell annotation

Adds projected type and source, replaces the type with `annot_2909.csv` and makes it track-safe. Writes `out_path("./data/md_metacells.csv")`.
- Caveat: the reannotation is assigned by row position, not joined on metacell name, so both tables must list metacells in the same order.

In [15]:
ad_project <- anndata::read_h5ad(paste0(base_dir,"data/dnmt_dko_brain_metacells_gotg95_projection.h5ad"))

In [32]:
md <- ad_project$obs %>%
    rownames_to_column("metacell") %>%
    select(metacell, cell_type = projected_type, source)
md <- md %>%
    mutate(cell_type_full = cell_type) %>%
    mutate(
        cell_type = gsub(" ", "_", cell_type), 
        cell_type = gsub("\\/", "_", cell_type)
    )

md_reannot = read_csv(paste0(embdnmt_dir, "/revision_e95/data/annot_2909.csv"))
md = md %>% mutate(reannot = md_reannot$cell_type,
                   cell_type = gsub(" ", "_", reannot), 
                   cell_type = gsub("\\/", "_", cell_type),
                   cell_type = gsub("\\+", "_", cell_type),
                   cell_type = gsub("[()]", "", cell_type),
                   cell_type = gsub("\\p{Pd}+", "_", cell_type, perl = TRUE),
                   cell_type = gsub("_+", "_", cell_type))

In [40]:
write.csv(md, out_path("./data/md_metacells.csv"))

## Per-state tracks

`mcc_to_cell_type_tracks` writes one 20 bp coverage track per cell type under `embdnmt.brain_e95_host.reannot` and `embdnmt.brain_e95_dko.reannot`.
- The `.reannot.marginal` tracks used later are not created here.

In [ ]:
mcc_host <- add_mc_metadata(mcc_host, md %>% filter(source == "host"))
mcc_dko <- add_mc_metadata(mcc_dko, md %>% filter(source == "dko"))

In [ ]:
gdir.create("embdnmt.brain_e95_host.reannot")
gdir.create("embdnmt.brain_e95_dko.reannot")
mcc_to_cell_type_tracks(mc_counts = mcc_host, track_prefix = "embdnmt.brain_e95_host.reannot")
mcc_to_cell_type_tracks(mc_counts = mcc_dko, track_prefix = "embdnmt.brain_e95_dko.reannot")

## Pooled lineage tracks

Host pools (the "late ectoderm" and "late mesoderm" groups of Fig. 6g), read by `E9.5_multiome.Fig6/Fig6.ipynb`:
- `Sox2_pool`: Dmbx1+ di/mesencephalon, optic vesicle/telencephalon, hindbrain, MHB, Hoxd3+ SCP
- `Twist1_pool`: somitic, sclerotome, Bmp4+ Igf2+ cranial muscles

Caveat: the DKO pool tracks exist in the database, but the code that made them is not here.

In [ ]:
gtrack.create(
  track = "embdnmt.brain_e95_host.reannot.Sox2_pool",
  description = "Pooled on all ectoderm states",
  expr = "embdnmt.brain_e95_host.reannot.Dmbx1_Diencephalon_Mesencephalon + embdnmt.brain_e95_host.reannot.Optic_vesicle_Telencephalon + embdnmt.brain_e95_host.reannot.Hindbrain + embdnmt.brain_e95_host.reannot.Midbrain_hindbrain_boundary + embdnmt.brain_e95_host.reannot.Hoxd3_spinal_cord_progenitors",
  iterator = NULL,
  band = NULL
)

In [ ]:
gtrack.create(
  track = "embdnmt.brain_e95_host.reannot.Twist1_pool",
  description = "Pooled on all mesoderm states",
  expr = "embdnmt.brain_e95_host.reannot.Somitic + embdnmt.brain_e95_host.reannot.Sclerotome + embdnmt.brain_e95_host.reannot.Bmp4_Igf2_Cranial_muscles",
  iterator = NULL,
  band = NULL
)

## Summed E9.5 + gastrulation DKO marginal

`embdnmt.brain_e95_dko.marginal_plus_gastr`: sum in 10 bp bins of the E9.5 host and DKO marginals and the gastrulation `mut_multi2.DNMT_DKO_{KO,host}.marginal`.
- Its `_regcap` normalization is made in `prep/E9.5_atac_peaks.ipynb`; `init_e95` (`E9.5_multiome.Fig6/Fig6_func.r`) calls the Fig. 6 CRE peaks on it.

In [ ]:
# Marginal of marginals #

gtrack.create(
  track = "embdnmt.brain_e95_dko.marginal_plus_gastr",
  description = "Summed marginal for E7.5/8.5/9.5 DNMT DKO and its host.",
  expr = "embdnmt.brain_e95_host.reannot.marginal + embdnmt.brain_e95_dko.reannot.marginal + mut_multi2.DNMT_DKO_KO.marginal + mut_multi2.DNMT_DKO_host.marginal",
  iterator = 10,
  band = NULL
)

In [11]:
gdb.reload()

In [13]:
gdb.reload()